# Graph Basics — Everything You Need Before LeetCode

Read top to bottom. By the end you will understand:
- What a graph is and its types
- How to store a graph in code (3 ways)
- BFS and DFS (the two algorithms that solve 90% of graph problems)
- Cycle detection, connected components, topological sort

---
## 1. What is a Graph?

A graph is a collection of **nodes (vertices)** connected by **edges**.

```
Nodes: cities
Edges: roads between cities

    A --- B
    |     |
    C --- D
```

Unlike a tree:
- A tree is a graph with no cycles and one root
- A graph can have **cycles**, **multiple components**, and **no root**

---
## 2. Types of Graphs

### Directed vs Undirected
```
Undirected:  A -- B   (road goes both ways)
Directed:    A --> B  (one-way street)
```

### Weighted vs Unweighted
```
Unweighted:  A -- B       (just connected)
Weighted:    A --5-- B    (edge has a cost/distance)
```

### Cyclic vs Acyclic
```
Cyclic:   A -> B -> C -> A   (you can loop back)
Acyclic:  A -> B -> C        (no loops)

DAG = Directed Acyclic Graph (very common in LeetCode)
```

### Connected vs Disconnected
```
Connected:     every node can reach every other node
Disconnected:  some nodes are isolated (separate islands)
```

---
## 3. How to Store a Graph in Code

### Method 1: Adjacency List (USE THIS — most common)

A dictionary where `key = node`, `value = list of neighbors`

```
Graph:
  0 -- 1
  |    |
  2 -- 3

Adjacency List:
  { 0: [1, 2],
    1: [0, 3],
    2: [0, 3],
    3: [1, 2] }
```

### Method 2: Adjacency Matrix

2D array where `matrix[i][j] = 1` means edge from i to j

```
     0  1  2  3
0  [ 0, 1, 1, 0 ]
1  [ 1, 0, 0, 1 ]
2  [ 1, 0, 0, 1 ]
3  [ 0, 1, 1, 0 ]
```
Use when: dense graph or O(1) edge lookup needed.

### Method 3: Edge List
Just a list of (u, v) pairs: `[(0,1), (0,2), (1,3), (2,3)]`
Use when: building the graph from input.

In [ ]:
from collections import defaultdict, deque
import heapq
from typing import List

# ---- Building an Adjacency List ----

# From edge list (undirected)
edges = [(0,1), (0,2), (1,3), (2,3)]
n = 4  # number of nodes

graph = defaultdict(list)
for u, v in edges:
    graph[u].append(v)
    graph[v].append(u)   # remove this line for directed graph

print("Adjacency List:")
for node in range(n):
    print(f"  {node} -> {graph[node]}")

---
## 4. BFS — Breadth First Search

Visit nodes **level by level** (nearest neighbors first).

Uses a **Queue (deque)**.

```
Graph:        BFS from node 0:
  0           Level 0: 0
 / \          Level 1: 1, 2
1   2         Level 2: 3, 4
|   |
3   4
```

**When to use BFS:**
- Shortest path in unweighted graph
- Level-order traversal
- "Minimum steps" problems
- "Nearest" problems

In [ ]:
def bfs(graph, start):
    visited = set([start])
    queue = deque([start])
    order = []

    while queue:
        node = queue.popleft()       # process front of queue
        order.append(node)

        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return order


# Test
edges = [(0,1), (0,2), (1,3), (2,4)]
graph = defaultdict(list)
for u, v in edges:
    graph[u].append(v)
    graph[v].append(u)

print("BFS order from node 0:", bfs(graph, 0))

### BFS with level tracking (needed for "minimum steps" problems)

In [ ]:
def bfs_levels(graph, start):
    visited = set([start])
    queue = deque([(start, 0)])    # (node, level)

    while queue:
        node, level = queue.popleft()
        print(f"node={node}  level={level}")

        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, level + 1))

bfs_levels(graph, 0)

---
## 5. DFS — Depth First Search

Go as **deep as possible** before backtracking.

Uses **recursion** (or a stack).

```
Graph:        DFS from node 0:
  0           0 -> 1 -> 3 (dead end, backtrack)
 / \            -> 2 -> 4
1   2
|   |
3   4
```

**When to use DFS:**
- Cycle detection
- Path finding (does path exist?)
- Connected components
- Topological sort
- Islands problems (grid DFS)

In [ ]:
# Recursive DFS
def dfs(graph, node, visited, order):
    visited.add(node)
    order.append(node)

    for neighbor in graph[node]:
        if neighbor not in visited:
            dfs(graph, neighbor, visited, order)


visited = set()
order = []
dfs(graph, 0, visited, order)
print("DFS order from node 0:", order)

In [ ]:
# Iterative DFS (using explicit stack)
def dfs_iterative(graph, start):
    visited = set()
    stack = [start]
    order = []

    while stack:
        node = stack.pop()            # process top of stack
        if node in visited:
            continue
        visited.add(node)
        order.append(node)

        for neighbor in graph[node]:
            if neighbor not in visited:
                stack.append(neighbor)

    return order

print("DFS iterative from node 0:", dfs_iterative(graph, 0))

---
## 6. BFS vs DFS — When to Use Which?

| Problem | Algorithm |
|---|---|
| Shortest path (unweighted) | BFS |
| Minimum steps / hops | BFS |
| Does a path exist? | DFS or BFS |
| Detect cycle | DFS |
| Count connected components | DFS or BFS |
| Topological sort | DFS |
| Number of islands | DFS (flood fill) |
| Shortest path (weighted) | Dijkstra (BFS + heap) |

---
## 7. Connected Components

A disconnected graph has multiple **islands** — groups of nodes with no path between groups.

```
0 -- 1    3 -- 4    6
     |         |
     2         5

3 connected components: {0,1,2}, {3,4,5}, {6}
```

In [ ]:
def count_components(n, edges):
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)

    visited = set()
    count = 0

    def dfs(node):
        visited.add(node)
        for nei in graph[node]:
            if nei not in visited:
                dfs(nei)

    for node in range(n):
        if node not in visited:     # new island found
            dfs(node)
            count += 1

    return count


edges = [(0,1),(1,2),(3,4),(4,5)]
print(count_components(7, edges))   # 3 components: {0,1,2}, {3,4,5}, {6}

---
## 8. Cycle Detection

### Undirected Graph — DFS
Track the parent. If you see a visited neighbor that is NOT your parent, there's a cycle.

In [ ]:
def has_cycle_undirected(n, edges):
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)

    visited = set()

    def dfs(node, parent):
        visited.add(node)
        for nei in graph[node]:
            if nei not in visited:
                if dfs(nei, node):
                    return True
            elif nei != parent:     # visited AND not parent = cycle!
                return True
        return False

    for node in range(n):
        if node not in visited:
            if dfs(node, -1):
                return True
    return False


print(has_cycle_undirected(4, [(0,1),(1,2),(2,0)]))   # True  (0-1-2-0)
print(has_cycle_undirected(4, [(0,1),(1,2),(2,3)]))   # False

### Directed Graph — DFS with rec_stack
Track nodes in the **current recursion path**. If you revisit a node still on the path, cycle exists.

In [ ]:
def has_cycle_directed(n, edges):
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)

    visited = set()
    rec_stack = set()   # nodes in current DFS path

    def dfs(node):
        visited.add(node)
        rec_stack.add(node)

        for nei in graph[node]:
            if nei not in visited:
                if dfs(nei):
                    return True
            elif nei in rec_stack:   # back edge = cycle
                return True

        rec_stack.remove(node)  # done with this path
        return False

    for node in range(n):
        if node not in visited:
            if dfs(node):
                return True
    return False


print(has_cycle_directed(3, [(0,1),(1,2),(2,0)]))   # True
print(has_cycle_directed(3, [(0,1),(1,2)]))          # False

---
## 9. Topological Sort (for DAGs)

Only works on **Directed Acyclic Graphs**.
Gives an ordering where every edge `u -> v` means `u` comes before `v`.

```
Courses:  Math -> Physics -> Engineering
          Math -> Chemistry

Valid order: Math, Chemistry, Physics, Engineering
```

### Method: Kahn's Algorithm (BFS-based)
- Compute **in-degree** (how many edges point INTO each node)
- Start with nodes that have in-degree 0 (no prerequisites)
- Remove them, reduce neighbors' in-degrees, repeat

In [ ]:
def topological_sort(n, edges):
    graph = defaultdict(list)
    in_degree = [0] * n

    for u, v in edges:
        graph[u].append(v)
        in_degree[v] += 1

    # Start with all nodes that have no prerequisites
    queue = deque([i for i in range(n) if in_degree[i] == 0])
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)

        for nei in graph[node]:
            in_degree[nei] -= 1
            if in_degree[nei] == 0:    # all prerequisites done
                queue.append(nei)

    # If order has all nodes, no cycle. Otherwise cycle exists.
    return order if len(order) == n else []


# 0=Math, 1=Physics, 2=Engineering, 3=Chemistry
edges = [(0,1),(1,2),(0,3)]
print("Topological order:", topological_sort(4, edges))   # [0, 1, 3, 2] or [0, 3, 1, 2]

---
## 10. Grid as a Graph (Islands problems)

A 2D grid is just a graph where each cell is a node and edges go to 4 neighbors (up, down, left, right).

```
Grid:         Graph edges (implicit):
1 1 0         (0,0)-(0,1), (0,0)-(1,0), (0,1)-(1,1) ...
1 1 0
0 0 1
```

**Pattern for grid DFS:**

In [ ]:
def numIslands(grid):
    if not grid:
        return 0

    rows, cols = len(grid), len(grid[0])
    count = 0

    def dfs(r, c):
        # Out of bounds or water or already visited
        if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] != '1':
            return
        grid[r][c] = '#'   # mark visited (flood fill)
        dfs(r+1, c)
        dfs(r-1, c)
        dfs(r, c+1)
        dfs(r, c-1)

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1':   # new island found
                dfs(r, c)
                count += 1

    return count


grid = [
    ['1','1','0','0','0'],
    ['1','1','0','0','0'],
    ['0','0','1','0','0'],
    ['0','0','0','1','1']
]
print("Number of islands:", numIslands(grid))   # 3

---
## 11. Shortest Path — BFS (Unweighted)

BFS guarantees the shortest path in an **unweighted** graph because it explores level by level.

In [ ]:
def shortest_path(n, edges, src, dst):
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)

    visited = set([src])
    queue = deque([(src, 0)])   # (node, distance)

    while queue:
        node, dist = queue.popleft()
        if node == dst:
            return dist
        for nei in graph[node]:
            if nei not in visited:
                visited.add(nei)
                queue.append((nei, dist + 1))

    return -1   # no path


edges = [(0,1),(0,2),(1,3),(2,3),(3,4)]
print("Shortest path 0->4:", shortest_path(5, edges, 0, 4))   # 2

---
## 12. Common LeetCode Graph Patterns

| Pattern | Algorithm | Key Idea |
|---|---|---|
| Number of islands | DFS/BFS on grid | Flood fill each island |
| Shortest path (unweighted) | BFS | Level = distance |
| Course schedule (possible?) | Topo sort / cycle detect | If cycle -> impossible |
| Clone graph | DFS + hashmap | Map old node to new node |
| Surrounded regions | DFS from border | Mark safe cells first |
| Rotten oranges | BFS (multi-source) | Start BFS from all rotten |
| Word ladder | BFS | Each word = node |
| Find if path exists | DFS/BFS/Union-Find | Simple traversal |
| Pacific Atlantic water flow | DFS from both coasts | Mark reachable cells |

---
## Summary Cheatsheet

```
GRAPH STORAGE
  defaultdict(list)  ->  adjacency list  (use this always)
  build: graph[u].append(v); graph[v].append(u)  (undirected)
         graph[u].append(v)                       (directed)

BFS  (queue, level by level)
  queue = deque([start]);  visited = {start}
  while queue: node = queue.popleft() -> process -> add unvisited neighbors
  USE FOR: shortest path, min steps, nearest

DFS  (recursion or stack, go deep first)
  visited.add(node) -> recurse on unvisited neighbors
  USE FOR: connected components, cycle detect, topo sort, islands

CYCLE DETECTION
  Undirected: DFS + track parent
  Directed:   DFS + rec_stack (current path)

TOPOLOGICAL SORT  (DAG only)
  Kahn's: in-degree array + BFS from 0-indegree nodes

GRID GRAPHS
  4 directions: (r+1,c),(r-1,c),(r,c+1),(r,c-1)
  Always check: bounds + condition + visited
```

Now open the LeetCode problems file — you have everything you need!